# 🏀 NBA Salary Predictor — Rede Neural v2

## Objetivo
Treinar uma Rede Neural com Keras/TensorFlow para prever
o salário anual de jogadores da NBA com base em suas
estatísticas de desempenho da temporada 2022-23.

## Contexto
Este notebook é uma evolução do modelo de Regressão Linear
desenvolvido no Módulo 1. A Rede Neural foi escolhida por:
- Capturar relações não lineares entre estatísticas e salário
- Arquiteturas mais flexíveis em relação a modelos clássicos
- Padrão da indústria para problemas preditivos mais complexos

## Dataset
- NBA Players Stats and Salaries 2022-23
- 467 jogadores, 52 colunas originais
- Fonte: Kaggle

In [ ]:
# ============================================================
# NBA Salary Predictor — Rede Neural
# Célula 1: Importações
# ============================================================

# Biblioteca padrão
import os
import json
import random
import warnings
from datetime import datetime

# Dados e visualização
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Keras/TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Serialização
import joblib

# Configurações
warnings.filterwarnings("ignore")

# Fixando seeds para reprodutibilidade
# (depois dos imports: os e tf precisam estar importados antes de serem usados)
os.environ["PYTHONHASHSEED"] = "42"
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow versão: {tf.__version__}")
print(f"✅ Keras versão: {keras.__version__}")
print("✅ Bibliotecas importadas com sucesso!")

## Fase 1: Carregamento dos Dados

Carregamos o mesmo dataset utilizado no Módulo 1 para
garantir uma comparação justa entre os modelos.
A variável alvo é `Salary` — valor numérico contínuo
em dólares representando o salário anual do jogador.

In [ ]:
# ============================================================
# Célula 2: Carregar e Preparar os Dados
# ============================================================
from pathlib import Path

NOME_CSV = "nba_2022-23_all_stats_with_salary.csv"

# Procura o CSV em data/ (raiz do projeto); se não achar, na pasta atual
caminho_csv = Path("data") / NOME_CSV
if not caminho_csv.exists():
    caminho_csv = Path(NOME_CSV)

# Google Colab: se o arquivo ainda não existir, pede o upload
if not caminho_csv.exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            f"Arquivo '{NOME_CSV}' não encontrado em data/ nem na pasta atual. "
            "Execute o notebook a partir da raiz do projeto."
        )

# Carregar o dataset
df = pd.read_csv(caminho_csv)

print(f"✅ Dataset carregado com sucesso!")
print(f"   Arquivo   : {caminho_csv}")
print(f"   Linhas    : {df.shape[0]}")
print(f"   Colunas   : {df.shape[1]}")
print(f"\n📋 Primeiros registros:")
print(df.head())

## Fase 2: Pré-processamento com Pipeline Scikit-learn

Reutilizamos o pipeline de pré-processamento do Módulo 1, com
**uma diferença importante: aqui não há winsorização de outliers**.
Os valores entram no modelo como estão no dataset, e a API
também os recebe sem tratamento, o que mantém treino e
inferência consistentes.

**Decisões técnicas:**
- **Pipeline Scikit-learn:** encadeia transformações evitando
  data leakage — o scaler é fitado apenas no treino
- **SimpleImputer (mediana):** distribuição assimétrica do
  salário torna a mediana mais robusta que a média
- **Label Encoding:** transforma posições em números
  mantendo apenas 1 coluna
- **Remoção de multicolinearidade:** 10 colunas removidas por
  alta correlação com outras variáveis (MP, 2P, FT, TOV, FG,
  DRB%, OWS, DWS, AST%, VORP)
- **Feature engineering:** `tier_time` (faixa salarial do time)
  e `eficiencia` (PTS + AST + TRB + STL + BLK)

In [ ]:
# ============================================================
# Célula 3: Pipeline de Pré-processamento com Scikit-learn
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

def preprocessar_dados(df):
    """
    Aplica o pipeline de pré-processamento completo:
    1. Remove colunas desnecessárias
    2. Simplifica posições e times combinados
    3. Remove duplicatas
    4. Feature Engineering
    5. Label Encoding
    6. Remove multicolinearidade
    Retorna o DataFrame processado e o LabelEncoder
    """
    df = df.copy()

    # --------------------------------------------------------
    # Passo 1: Remover colunas desnecessárias
    # --------------------------------------------------------
    colunas_remover = [
        "Player Name", "Unnamed: 0",
        "FG%", "3P%", "FT%", "2P%", "eFG%",
        "FGA", "3PA", "FTA", "2PA",
        "BPM", "WS/48", "OBPM", "DBPM",
        "ORB", "DRB", "Total Minutes"
    ]
    df = df.drop(columns=colunas_remover)

    # --------------------------------------------------------
    # Passo 2: Simplificar posições e times combinados
    # --------------------------------------------------------
    df["Position"] = df["Position"].apply(
        lambda x: x.split("-")[0] if "-" in str(x) else x
    )
    df["Team"] = df["Team"].apply(
        lambda x: x.split("/")[0] if "/" in str(x) else x
    )

    # --------------------------------------------------------
    # Passo 3: Remover duplicatas
    # --------------------------------------------------------
    df = df.drop_duplicates()

    # --------------------------------------------------------
    # Passo 4: Feature Engineering
    # --------------------------------------------------------
    salary_por_time = df.groupby("Team")["Salary"].mean()
    p33 = salary_por_time.quantile(0.33)
    p66 = salary_por_time.quantile(0.66)

    def classificar_tier(team):
        salario_medio = salary_por_time[team]
        if salario_medio >= p66:
            return 3
        elif salario_medio >= p33:
            return 2
        else:
            return 1

    df["tier_time"] = df["Team"].apply(classificar_tier)
    df = df.drop(columns=["Team"])
    df["eficiencia"] = (
        df["PTS"] + df["AST"] +
        df["TRB"] + df["STL"] + df["BLK"]
    )

    # --------------------------------------------------------
    # Passo 5: Label Encoding
    # --------------------------------------------------------
    le = LabelEncoder()
    df["Position"] = le.fit_transform(df["Position"])

    # --------------------------------------------------------
    # Passo 6: Remover multicolinearidade
    # --------------------------------------------------------
    colunas_remover_corr = [
        "MP", "2P", "FT", "TOV", "FG",
        "DRB%", "OWS", "DWS", "AST%", "VORP"
    ]
    colunas_remover_corr = [c for c in colunas_remover_corr if c in df.columns]
    df = df.drop(columns=colunas_remover_corr)

    return df, le

# Aplicar pré-processamento
df_processado, le = preprocessar_dados(df)

# --------------------------------------------------------
# Separando X e y
# --------------------------------------------------------
X = df_processado.drop(columns=["Salary"])
y = df_processado["Salary"]

# --------------------------------------------------------
# Divisão 80/20 — treino e teste
# --------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --------------------------------------------------------
# Pipeline de escalonamento
# Garante que o scaler é fitado apenas no treino
# e aplicado de forma consistente no teste
# --------------------------------------------------------
pipeline_scaler = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # trata NaN residuais
    ("scaler", StandardScaler())                     # padronização
])

X_train_scaled = pipeline_scaler.fit_transform(X_train)
X_test_scaled  = pipeline_scaler.transform(X_test)

# Divisão para validação durante treino da Rede Neural
X_train_real, X_val, y_train_real, y_val = train_test_split(
    X_train_scaled, y_train,
    test_size=0.2,
    random_state=42
)

print("=== PIPELINE APLICADO COM SUCESSO ===")
print("-" * 40)
print(f"  Total de registros : {len(X)}")
print(f"  Treino real        : {len(X_train_real)} registros")
print(f"  Validação          : {len(X_val)} registros")
print(f"  Teste              : {len(X_test_scaled)} registros")
print(f"  Features           : {X.shape[1]}")
print("-" * 40)

# Salvando o pipeline para usar na API
os.makedirs("models", exist_ok=True)
joblib.dump(pipeline_scaler, "models/pipeline_scaler.pkl")
print("\n✅ Pipeline salvo em models/pipeline_scaler.pkl")

## Fase 3: Arquitetura da Rede Neural

**Decisões técnicas:**
- **128-64-32 neurônios:** redução gradual para aprendizado
  hierárquico — da camada mais ampla para a mais específica
- **ReLU:** evita o problema de gradiente desaparecendo
  em comparação com Sigmoid e Tanh
- **Dropout 20%:** desativa aleatoriamente 20% dos neurônios
  durante o treino para evitar overfitting
- **Saída linear:** necessário para regressão — sem limitação
  de valores permite prever qualquer salário
- **Otimizador Adam:** adaptativo e eficiente — padrão
  da indústria para redes neurais
- **Loss MSE:** penaliza erros grandes — adequado para
  regressão de valores monetários

In [ ]:
# ============================================================
# Célula 4: Construção da Rede Neural com Keras
# ============================================================

def criar_modelo_rn(input_dim):
    """
    Arquitetura da Rede Neural para regressão de salários:
    - 3 camadas densas com ativação ReLU
    - Dropout para evitar overfitting
    - Camada de saída linear para regressão
    """
    modelo = Sequential([
        # --------------------------------------------------------
        # Camada 1: aprende padrões complexos das 24 features
        # --------------------------------------------------------
        Dense(128, activation="relu", input_dim=input_dim,
              name="camada_1"),
        Dropout(0.2, name="dropout_1"),  # desativa 20% dos neurônios

        # --------------------------------------------------------
        # Camada 2: refina os padrões aprendidos
        # --------------------------------------------------------
        Dense(64, activation="relu", name="camada_2"),
        Dropout(0.2, name="dropout_2"),

        # --------------------------------------------------------
        # Camada 3: consolida os padrões
        # --------------------------------------------------------
        Dense(32, activation="relu", name="camada_3"),

        # --------------------------------------------------------
        # Camada de saída: 1 neurônio com ativação linear
        # Linear pois estamos prevendo um valor contínuo
        # --------------------------------------------------------
        Dense(1, activation="linear", name="saida")
    ])

    # --------------------------------------------------------
    # Compilação do modelo
    # adam    : otimizador adaptativo — padrão para redes neurais
    # mse     : loss para regressão — penaliza erros grandes
    # mae     : métrica de acompanhamento — fácil de interpretar
    # --------------------------------------------------------
    modelo.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return modelo

# Criar o modelo
modelo_rn = criar_modelo_rn(input_dim=X_train_real.shape[1])

# Exibir resumo da arquitetura
print("=== ARQUITETURA DA REDE NEURAL ===")
modelo_rn.summary()

## Fase 4: Treinamento

**Decisões técnicas:**
- **200 épocas máximo:** suficiente para convergência
  com EarlyStopping controlando o ponto de parada
- **Batch size 32:** equilíbrio entre velocidade e
  estabilidade do gradiente
- **EarlyStopping (patience=20):** evita overfitting
  parando quando val_loss para de melhorar
- **ReduceLROnPlateau:** reduz a taxa de aprendizado
  pela metade quando o modelo fica estagnado e permite
  refinamentos mais precisos nos pesos
- **Conjunto de validação:** 20% do treino separado
  para monitorar overfitting durante o treinamento

In [ ]:
# ============================================================
# Célula 5: Treinamento da Rede Neural
# ============================================================

# --------------------------------------------------------
# Callbacks — funções executadas durante o treino
# --------------------------------------------------------

# EarlyStopping: para o treino quando o modelo
# para de melhorar evitando overfitting
early_stopping = EarlyStopping(
    monitor="val_loss",        # monitora o erro na validação
    patience=20,               # aguarda 20 épocas sem melhora
    restore_best_weights=True, # restaura os melhores pesos
    verbose=1
)

# ReduceLROnPlateau: reduz a taxa de aprendizado
# quando o modelo para de melhorar
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",  # monitora o erro na validação
    factor=0.5,          # reduz o LR pela metade
    patience=10,         # aguarda 10 épocas sem melhora
    min_lr=1e-6,         # limite mínimo do LR
    verbose=1
)

# --------------------------------------------------------
# Treinamento
# --------------------------------------------------------
print("=== INICIANDO TREINAMENTO ===")
print("-" * 40)

historico = modelo_rn.fit(
    X_train_real, y_train_real,
    epochs=200,              # máximo de épocas
    batch_size=32,           # amostras por atualização
    validation_data=(X_val, y_val),  # dados de validação
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("\n✅ Treinamento concluído!")
print(f"   Épocas treinadas: {len(historico.history['loss'])}")

## Fase 5: Análise do Histórico de Treinamento

Visualizamos as curvas de aprendizado para diagnosticar:
- **Convergência:** as curvas devem descer e estabilizar
- **Overfitting:** treino muito melhor que validação indica
  memorização dos dados
- **Underfitting:** ambas as curvas altas indicam que o
  modelo não aprendeu os padrões

In [ ]:
# ============================================================
# Célula 6: Visualização do Histórico de Treinamento
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Histórico de Treinamento — Rede Neural",
             fontsize=14, fontweight="bold")

# --------------------------------------------------------
# Gráfico 1: Loss (MSE) — Treino vs Validação
# --------------------------------------------------------
axes[0].plot(historico.history["loss"],
             label="Treino", color="steelblue", linewidth=2)
axes[0].plot(historico.history["val_loss"],
             label="Validação", color="orange", linewidth=2)
axes[0].set_title("Loss (MSE) — Treino vs Validação")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(True)

# --------------------------------------------------------
# Gráfico 2: MAE — Treino vs Validação
# --------------------------------------------------------
axes[1].plot(historico.history["mae"],
             label="Treino", color="steelblue", linewidth=2)
axes[1].plot(historico.history["val_mae"],
             label="Validação", color="orange", linewidth=2)
axes[1].set_title("MAE — Treino vs Validação")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("MAE (US$)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("grafico_historico_treino.png", dpi=120)
plt.show()
plt.close()
print("✅ Gráfico de histórico gerado!")

## Fase 6: Avaliação e Comparativo dos Modelos

Avaliamos a Rede Neural no conjunto de teste e comparamos com
a Regressão Linear e o KNN (K=10), **treinados aqui, nos mesmos
dados e no mesmo split**. Assim a comparação é justa: os valores
do Módulo 1 usavam outliers limitados e não são diretamente
comparáveis com os desta base.

**Métricas utilizadas:**
- **MAE:** erro médio em dólares — fácil interpretação
- **RMSE:** penaliza erros grandes — critério de desempate
- **R²:** percentual da variação explicada pelo modelo

O conjunto de teste tem poucos registros (20% da base), então
diferenças pequenas entre modelos podem ser apenas ruído.

In [ ]:
# ============================================================
# Célula 7: Avaliação e Comparativo dos Modelos
# ============================================================

def calcular_metricas(y_real, y_pred):
    """MAE, MSE, RMSE e R² de um conjunto de previsões."""
    mse = mean_squared_error(y_real, y_pred)
    return {
        "MAE" : float(mean_absolute_error(y_real, y_pred)),
        "MSE" : float(mse),
        "RMSE": float(np.sqrt(mse)),
        "R2"  : float(r2_score(y_real, y_pred)),
    }

# --------------------------------------------------------
# Rede Neural (modelo treinado com early stopping na Fase 4)
# --------------------------------------------------------
y_pred_rn = modelo_rn.predict(X_test_scaled, verbose=0).flatten()
metricas_rn = calcular_metricas(y_test, y_pred_rn)

print("=== AVALIAÇÃO DA REDE NEURAL ===")
print("-" * 45)
print(f"  MAE  : US$ {metricas_rn['MAE']:,.2f}")
print(f"  MSE  : US$ {metricas_rn['MSE']:,.2f}")
print(f"  RMSE : US$ {metricas_rn['RMSE']:,.2f}")
print(f"  R²   : {metricas_rn['R2']:.4f}")
print("-" * 45)

# --------------------------------------------------------
# Modelos clássicos: treinados AQUI, nos mesmos dados
# (X_train_scaled / X_test_scaled) e no mesmo split
# --------------------------------------------------------
modelo_lr  = LinearRegression().fit(X_train_scaled, y_train)
modelo_knn = KNeighborsRegressor(n_neighbors=10).fit(X_train_scaled, y_train)

metricas_lr  = calcular_metricas(y_test, modelo_lr.predict(X_test_scaled))
metricas_knn = calcular_metricas(y_test, modelo_knn.predict(X_test_scaled))

# --------------------------------------------------------
# Comparativo (todos os valores calculados, nenhum digitado)
# --------------------------------------------------------
resultados = {
    "Regressão Linear": metricas_lr,
    "KNN (K=10)"      : metricas_knn,
    "Rede Neural"     : metricas_rn,
}

print("\n=== COMPARATIVO DOS MODELOS (mesmos dados, mesmo split) ===")
print("-" * 65)
print(f"  {'Modelo':<25} {'MAE':>15} {'RMSE':>15} {'R²':>8}")
print("-" * 65)
for nome, m in resultados.items():
    print(f"  {nome:<25} US$ {m['MAE']:>11,.0f} US$ {m['RMSE']:>11,.0f} {m['R2']:>8.4f}")
print("-" * 65)

menor_rmse = min(resultados, key=lambda nome: resultados[nome]["RMSE"])
print(f"\n📌 Menor RMSE no teste: {menor_rmse}")
if menor_rmse != "Rede Neural":
    print("   ⚠️ A Rede Neural não teve o menor RMSE nesta base. Ela continua sendo o")
    print("      modelo serializado (objetivo do projeto); registre essa comparação no relatório.")

## Fase 7: Serialização do Modelo

Retreinamos a Rede Neural no **conjunto de treino completo**
(80% da base, 373 registros), **mantendo o teste isolado**.
O número de épocas é o do melhor ponto do EarlyStopping
(menor `val_loss`). O modelo salvo é então avaliado no
conjunto de teste, e essas são as métricas gravadas no JSON.

**Arquivos salvos:**
- `modelo_rn_v2.keras` — modelo Keras no formato nativo
- `pipeline_scaler.pkl` — pipeline de pré-processamento
- `label_encoder.pkl` — encoder das posições
- `metricas_v2.json` — métricas e metadados do modelo

**Por que formato .keras ao invés de .pkl?**
O formato nativo do Keras preserva a arquitetura completa,
os pesos e a configuração do otimizador, mais seguro
e portátil que serializar com joblib.

In [ ]:
# ============================================================
# Célula 8: Serialização do Modelo
# ============================================================

# --------------------------------------------------------
# Retreinando no conjunto de TREINO completo (80% da base)
# O teste continua isolado: o modelo nunca o vê no treino.
# Épocas = melhor ponto do EarlyStopping (menor val_loss)
# --------------------------------------------------------
print("=== RETREINANDO NO CONJUNTO DE TREINO COMPLETO ===")
print("-" * 45)

melhor_epoca = int(np.argmin(historico.history["val_loss"])) + 1

# Recriando o modelo com a mesma arquitetura
modelo_final = criar_modelo_rn(input_dim=X_train_scaled.shape[1])

historico_final = modelo_final.fit(
    X_train_scaled, y_train,
    epochs=melhor_epoca,
    batch_size=32,
    verbose=0                # sem output para ficar mais limpo
)
print(f"✅ Modelo retreinado com {len(X_train_scaled)} registros de treino ({melhor_epoca} épocas)")

# --------------------------------------------------------
# Avaliando o modelo que SERÁ SALVO, no teste isolado
# (as métricas da Célula 7 são do modelo_rn, outro modelo)
# --------------------------------------------------------
y_pred_final = modelo_final.predict(X_test_scaled, verbose=0).flatten()
metricas_final = calcular_metricas(y_test, y_pred_final)

print("\n=== AVALIAÇÃO DO MODELO SALVO (teste isolado) ===")
print(f"  MAE  : US$ {metricas_final['MAE']:,.2f}")
print(f"  RMSE : US$ {metricas_final['RMSE']:,.2f}")
print(f"  R²   : {metricas_final['R2']:.4f}")

# --------------------------------------------------------
# Salvando o modelo no formato nativo do Keras
# --------------------------------------------------------
os.makedirs("models", exist_ok=True)
modelo_final.save("models/modelo_rn_v2.keras")
print("\n✅ Modelo salvo em models/modelo_rn_v2.keras")

# --------------------------------------------------------
# Salvando o pipeline de pré-processamento
# --------------------------------------------------------
joblib.dump(pipeline_scaler, "models/pipeline_scaler.pkl")
print("✅ Pipeline salvo em models/pipeline_scaler.pkl")

# --------------------------------------------------------
# Salvando o LabelEncoder
# --------------------------------------------------------
joblib.dump(le, "models/label_encoder.pkl")
print("✅ LabelEncoder salvo em models/label_encoder.pkl")

# --------------------------------------------------------
# Salvando as métricas em JSON
# --------------------------------------------------------
def arredondar(m):
    return {k: round(v, 4 if k == "R2" else 2) for k, v in m.items()}

metricas_v2 = {
    "modelo"              : "Rede Neural (Keras)",
    "versao"              : "v2",
    "data_treinamento"    : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset"             : "NBA Players Stats and Salaries 2022-23",
    "n_registros_total"   : len(df_processado),
    "n_registros_treino"  : len(X_train_scaled),
    "n_registros_teste"   : len(X_test_scaled),
    "arquitetura"         : "128-64-32-1 (ReLU + Dropout 20%)",
    "epochs_treinadas"    : melhor_epoca,
    "batch_size"          : 32,
    "optimizer"           : "adam",
    "variaveis_preditoras": X.columns.tolist(),
    "comparativo_modelos" : {
        "Regressao_Linear": arredondar(metricas_lr),
        "KNN_K10"         : arredondar(metricas_knn),
        "Rede_Neural"     : arredondar(metricas_final),
    },
    "metricas_teste"      : arredondar(metricas_final),
    "observacao"          : (
        "Rede_Neural e metricas_teste são do modelo salvo (retreinado no treino completo), "
        "avaliado no teste isolado. Regressão Linear e KNN foram treinados nos mesmos dados e split."
    ),
}

with open("models/metricas_v2.json", "w", encoding="utf-8") as f:
    json.dump(metricas_v2, f, indent=2, ensure_ascii=False)
print("✅ Métricas salvas em models/metricas_v2.json")

print("\n📄 Métricas registradas:")
print(json.dumps(metricas_v2, indent=2, ensure_ascii=False))